# Example visualizations with `altair`, `geopandas`, and `great_tables`.

There are generally 3 types of visualizations we want, and here are 3 Python packages we use to achieve what we want. :
1. charts for tabular data
   - `altair` can be interactive
   - `matplotlib` or `seaborn` (built on top of `matplotlib`) are not interactive, but syntax can allow for quick and easy charts to be made
   - `plotly` can be interactive, though embedding it in our static HTML sites is a challenge. We don't use a `Dash` app to keep our sites up.
   - `voila` takes Jupyter notebooks and creates interactive widgets to use on a dashboard. The challenge has been getting the dashboard to be "sendable" to others, as your local dashboard might be interactive, but we don't have an app to keep the dashboard alive.

2. geospatial data on a map
   - `geopandas` has its own way to display interactive maps and static maps
   - `folium` is used under-the-hood of `geopandas`, though using `folium` directly also gets you interactive maps
   - `ipyleaflet` is also for interactive maps, and is based on Leaflet in Javascript

3. tables of summary stats - can have charts embedded within columns of table
   - can use `pandas.describe()` for basic look
   - `pandas-profiling` looks at the df and gives basic descriptive stats for early exploratory work
   -  `great_tables` has been the best for embedding charts within tables. use for publication-ready tables, rather than exploring what's in a df.

In [ ]:
import geopandas as gpd
import google.auth
import pandas as pd

from shared_vars import INTERMED_GCS

credentials, _ = google.auth.default()

In [ ]:
gdf = gpd.read_parquet(
    f"{INTERMED_GCS}dim_stops_with_feed_service_period.parquet", 
    storage_options={"token": credentials.token},
    filters = [[
        ("schedule_name", "in", ["Gold Coast Schedule", "San Diego Schedule"])
    ]],
    columns = ["schedule_name", "feed_key", "stop_id", "geometry"]
)

## `geopandas` 

`geopandas` has the ability to plot interactive and static maps, usually we always want interative.
* timestamps, dates, arrays, lists in columns etc will cause an error, so remove those columns
* https://geopandas.org/en/stable/docs/user_guide/interactive_mapping.html
* [example with multiple layers](https://github.com/cal-itp/data-analyses/blob/main/rt_predictions/operator_report.ipynb)

In [ ]:
gdf.head()

In [ ]:
gdf[["stop_id", "geometry"]].explore()

In [ ]:
gdf.explore(
    "feed_key", categorical=True, legend=False
)

In [ ]:
gdf.explore("stop_id", categorical=True, tiles = "CartoDB Positron")

In [ ]:
gdf.plot()

In [ ]:
gdf2 = gdf.groupby(["stop_id", "geometry"]).agg({"feed_key": "nunique"}).reset_index()
gdf2 = gpd.GeoDataFrame(gdf2, geometry="geometry")
gdf2.explore("stop_id", legend=True, tiles="CartoDB DarkMatter")

## `altair`

`altair` is one of many plotting libraries in Python. It can add interactivity to charts, making it one of the preferred plotting packages. It has quite a large range of chart types. If there's not a type of chart in `altair`, look in `matplotlib` or `seaborn`.
* can do some basic geospatial ones, but typically in bar charts, line charts, you also can't have complex data types (geometry, arrays, etc), but timestamps are possible because time-series data can be plotted
* Example gallery: https://altair-viz.github.io/gallery/index.html)
* [examples with RT operators](https://github.com/cal-itp/data-analyses/blob/main/rt_predictions/chart_utils_for_operators.py)
and [RT stops](https://github.com/cal-itp/data-analyses/blob/main/rt_predictions/chart_utils_for_stops.py)
* [Curriculum](https://github.com/cal-itp/data-analyses/tree/6fd6e875dd2dfab134ac18ce7e54bb7068462730/starter_kit), more geopandas work that's not just plotting

In [ ]:
import altair as alt
import gcsfs
import pandas as pd

In [ ]:
stops_by_operator = gdf.groupby(["schedule_name"]).agg({
    "feed_key": "nunique",
    "stop_id": "count"
}).reset_index()

In [ ]:
(alt.Chart(stops_by_operator)
 .mark_bar()
 .encode(
     x="schedule_name",
     y="stop_id"
 )
)


## `great_tables`

* Tables are commonly displayed, and `great_tables` and `gt_extras` allow you to layer visualizations within a column of the table.
* Great table examples: https://posit-dev.github.io/great-tables/examples/
* Great table extras (more advanced plotting, must use with great tables): https://posit-dev.github.io/gt-extras/examples/
* [very simple table](https://github.com/cal-itp/data-analyses/blob/main/gtfs_digest/report_legislative_district.ipynb)
* [nanoplots](https://github.com/cal-itp/data-analyses/blob/main/rt_predictions/stop_report.ipynb) - must use polars and create columns that are arrays to be plotted
* [RT operator report uses lots of gt extras](https://github.com/cal-itp/data-analyses/blob/main/rt_predictions/operator_report.ipynb)
* If you just want to embed an interactive, searchable table (with no charting), you can use `itables`.

In [ ]:
from great_tables import GT
import gt_extras as gte

In [ ]:
(GT(stops_by_operator)
 .cols_label(
     schedule_name = "Operator",
     feed_key="n feeds",
     stop_id="# distinct stops"
 ).fmt_integer(["feed_key", "stop_id"])
 .tab_header(
        title="Summary Stats",
        subtitle="Source: dim_stops"
    )
)